In [1]:
import os
import re
import torch
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

In [3]:


# =========================================================
# DATASET PATH
# =========================================================

DATASET_PATH = "/content/drive/MyDrive/nlp_project/data/raw"

# =========================================================
# STORAGE
# =========================================================

data = []

article_id = 0

# =========================================================
# READ DATASET
# =========================================================

for category in os.listdir(DATASET_PATH):

    folder_path = os.path.join(
        DATASET_PATH,
        category
    )

    # kiểm tra có phải folder không
    if os.path.isdir(folder_path):

        for file_name in os.listdir(folder_path):

            # chỉ đọc file txt
            if file_name.endswith(".txt"):

                file_path = os.path.join(
                    folder_path,
                    file_name
                )

                try:

                    with open(
                        file_path,
                        "r",
                        encoding="utf-8"
                    ) as f:

                        raw_text = f.read()

                    # =================================================
                    # EXTRACT TITLE
                    # =================================================

                    title_match = re.search(
                        r"TIÊU ĐỀ:\s*(.*)",
                        raw_text
                    )

                    title = (
                        title_match.group(1).strip()
                        if title_match
                        else ""
                    )

                    # =================================================
                    # EXTRACT TIME
                    # =================================================

                    time_match = re.search(
                        r"THỜI GIAN:\s*(.*)",
                        raw_text
                    )

                    time = (
                        time_match.group(1).strip()
                        if time_match
                        else ""
                    )

                    # =================================================
                    # EXTRACT SOURCE URL
                    # =================================================

                    source_match = re.search(
                        r"NGUỒN:\s*(.*)",
                        raw_text
                    )

                    source_url = (
                        source_match.group(1).strip()
                        if source_match
                        else ""
                    )

                    # =================================================
                    # EXTRACT DOMAIN
                    # =================================================

                    domain_match = re.search(
                        r"https?://([^/]+)",
                        source_url
                    )

                    source = (
                        domain_match.group(1)
                        if domain_match
                        else ""
                    )

                    # =================================================
                    # EXTRACT MAIN CONTENT
                    # =================================================

                    content_split = raw_text.split(
                        "--------------------------------------------------"
                    )

                    if len(content_split) > 1:

                        content = content_split[1].strip()

                    else:

                        content = raw_text

                    # =================================================
                    # CLEAN CONTENT
                    # =================================================

                    content = re.sub(
                        r"\s+",
                        " ",
                        content
                    ).strip()

                    # =================================================
                    # CREATE RECORD
                    # =================================================

                    article_id += 1

                    data.append({

                        "id": article_id,

                        "title": title,

                        "text": content,

                        "category": category,

                        "source": source,

                        "url": source_url,

                        "time": time,

                        "file_name": file_name
                    })

                except Exception as e:

                    print(f"Error reading {file_name}: {e}")

# =========================================================
# CREATE DATAFRAME
# =========================================================

df = pd.DataFrame(data)

# =========================================================
# SHOW DATA
# =========================================================

print(df.head())

print("\nDataset Shape:")

print(df.shape)

print("\nColumns:")

print(df.columns)

   id                                              title  \
0   1  Sản vật miền Tây có tên gọi dễ nhầm lẫn, thịt ...   
1   2  Cụ ông U90 du lịch bụi khắp thế giới suốt 10 n...   
2   3  Tín đồ du lịch Hàn Quốc chuộng ‘mua trả góp’ n...   
3   4  Địa điểm du lịch gần Hà Nội dịp tết Bính Ngọ 2...   
4   5  Cận tết Nguyên đán, đến Lạng Sơn trải nghiệm l...   

                                                text category         source  \
0  Tuy có tên gọi dễ khiến người nghe nhầm lẫn nh...  du_lich  vietnamnet.vn   
1  TRUNG QUỐC - Ở tuổi 82, ông Dư Long Thái, một ...  du_lich  vietnamnet.vn   
2  Mua trả góp trở thành lựa chọn của nhiều du kh...  du_lich  vietnamnet.vn   
3  Dưới đây là gợi ý 3 địa điểm du lịch có khung ...  du_lich  vietnamnet.vn   
4  Lễ hội hoa đào xứ Lạng 2026 sẽ diễn ra với chủ...  du_lich  vietnamnet.vn   

                                                 url  \
0  https://vietnamnet.vn/san-vat-mien-tay-co-ten-...   
1  https://vietnamnet.vn/cu-ong-u90-du-lich-bu

In [4]:
df.isnull().sum()

,0
id,0
title,0
text,0
category,0
source,0
url,0
time,0
file_name,0


In [5]:
df["category"].value_counts()

,count
category,
the_gioi,1068
phap_luat,956
xa_hoi,806
kinh_te,596
y_te,577
thoi_su,572
the_thao,550
truyen_hinh,495
du_lich,356


In [6]:
df["text_length"] = df["text"].apply(len)

print(df["text_length"].describe())

count      6256.000000
mean       3625.666880
std        6768.802545
min           0.000000
25%        1666.000000
50%        2458.500000
75%        3851.250000
max      337615.000000
Name: text_length, dtype: float64


In [7]:
df.to_csv(

    "/content/drive/MyDrive/nlp_project/data/processed/news_dataset.csv",

    index=False,

    encoding="utf-8-sig"
)

print("Dataset saved!")

Dataset saved!
